# 🏌️ NeuroGolf **7186.61 LB** — fork-ready bundle & improvement workbench · [`github qurore/kaggloop`](https://github.com/qurore/kaggloop)

**Press *Copy & Edit* → *Run All* → *Submit*** and you get a complete, validated `submission.zip` (400 ONNX nets) that scores **7186.61** on the public LB — 🥉 bronze, above every public bundle.

**The 3-step improvement loop this notebook gives you:**
1. **Find a target** — Section 5 prints the per-task **cost table**. Score is `max(1, 25 − ln(params+bytes))`, so it is *logarithmic*: your few most expensive tasks are worth 10× the cheap ones. Attack those.
2. **Drop in your improved net** — paste it into the `OVERRIDES` cell (base64) *or* attach any Kaggle dataset containing `taskNNN.onnx` files. Nothing else to wire.
3. **Verify before you spend a submission** — Section 3 scores your override with the **official** `neurogolf_utils` (the grader's own sanitizer + ORT-1.24.4 profiler) and **refuses to package anything that fails** — protecting your daily submissions.

Built by autonomous agents running [**kaggloop**](https://github.com/qurore/kaggloop) — an open-source, Claude-Code-native competition loop (scout → survey → hypothesize → experiment → submit, with data-leakage + fresh-seed verification gates). This bundle = community shared-plateau nets + agent-crafted circuits + verified per-task min-merges + a **generator-proven canvas-crop surgery** (new in this version, see below).

## Why these circuits are smaller (5 server-validated trick families)

Scorer facts: `params` = initializer **element count** · `memory` = **intermediate** tensor bytes only · `input`/`output` tensors & node *attributes* are **free** · single-node graphs have zero memory. So the whole game is **fewer, smaller intermediates** — not smaller weights.

| # | Trick | Verified win |
|---|-------|--------------|
| 1 | **Initializer reuse is free** — one initializer feeding N inputs is counted once ⇒ rank-factored Einsum sandwich (`'ra,ai,zcij,bj,sb->zcrs'`, U/S used twice) = `2·30·k` params for any rank-k remap | task108: 971 → **300** |
| 2 | **`value_info` legalizes data-dependent Slice/Pad** — attach static value_info to dynamic-origin windows; checker + profiler accept ⇒ bbox crops straight off the free `input` | task014: 7985 → **4171** |
| 3 | **Terminal `GridSample`** — fp16 `[1,30,30,2]` grid (legal with an fp32 input) = gather + mask + zero-pad in one free node | task029: 10770 → **5606** |
| 4 | **`ConvInteger` renderer** — u8 codes + `x_zero_point=1` ⇒ signed weights & background-decoding pads; i32 output graded by `>0` | task031: 1472 → **688** |
| 5 | **Canvas-crop surgery** *(new)* — when the task's generator provably never exceeds `N×N` (read the [arc-gen source](https://github.com/google/arc-gen) or sample ≥10⁴ draws), shrink the whole 30×30 working canvas to `N×N`: crop right after the input consumers, rewrite every canvas-encoding constant (30→N, 900→N², …), and re-inflate once before the terminal | task018: 38247 → **33865** |

**+ verified min-merge:** every public per-task net is re-scored with the official grader and kept only where it is *strictly cheaper* and passes verification — a net that overfits the shipped demos (the classic local-pass → LB-0 trap) is rejected before it can zero your submission.

Traps to avoid: `sparse_initializer` (the official sanitizer leaves dangling refs → load failure) and **unproven static window sizes** (bounds observed from the few local demos can still fail the hidden suite — prove them from the generator or ≥10⁴ draws).

In [ ]:
import importlib.util, platform, sys
print("Python:", sys.version.split()[0], "|", platform.platform())
for p in ("onnx", "onnxruntime", "numpy"):
    print(p, "available:", importlib.util.find_spec(p) is not None)


In [ ]:
# ============ 1) The validated 7185.60 bundle -> submission.zip ============
import base64, hashlib, io, zipfile
B64 = "..."
raw = base64.b64decode(B64)
print("bundle bytes:", len(raw), "sha256:", hashlib.sha256(raw).hexdigest()[:16])
MODELS = {}
with zipfile.ZipFile(io.BytesIO(raw)) as z:
    for nm in z.namelist():
        MODELS[nm] = z.read(nm)
print("loaded", len(MODELS), "task nets (expect 400)")


In [ ]:
# ============ 2) YOUR IMPROVEMENTS GO HERE ============
# Option A: paste base64-encoded onnx bytes per task:
#   OVERRIDES = {"task123.onnx": "AAAA...=="}
# Option B: attach any Kaggle dataset containing taskNNN.onnx files
#   (+ Add Input -> your dataset). They are picked up automatically.
import base64, glob, os

OVERRIDES = {
    # "task123.onnx": "<base64 of your improved model>",
}

pending = {}
for name, b in OVERRIDES.items():
    pending[name] = base64.b64decode(b)
for path in glob.glob("/kaggle/input/*/task[0-9][0-9][0-9].onnx"):
    name = os.path.basename(path)
    with open(path, "rb") as f:
        pending[name] = f.read()
print(f"overrides staged: {sorted(pending) if pending else 'none (base bundle will be submitted)'}")


In [ ]:
# ============ 3) OFFICIAL local verification of your overrides ============
# Installs onnxruntime (same version as the grader: 1.24.4) if missing, loads the
# competition's own neurogolf_utils, and scores each staged override on ALL
# examples. Failing overrides are rejected -- protecting your daily submissions.
import importlib, importlib.util, io, json, math, os, subprocess, sys, tempfile, zipfile

if importlib.util.find_spec("onnxruntime") is None:
    print("installing onnxruntime==1.24.4 (requires Internet ON in notebook settings)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime==1.24.4"],
                       capture_output=True, text=True)
    importlib.invalidate_caches()
    if importlib.util.find_spec("onnxruntime") is None:
        print("onnxruntime unavailable (no internet?) -- overrides will be packaged UNVERIFIED")

UTILS = None
if importlib.util.find_spec("onnxruntime") is not None:
    for cand in ("/kaggle/input/competitions/neurogolf-2026/neurogolf_utils/neurogolf_utils.py",
                 "/kaggle/input/neurogolf-2026/neurogolf_utils/neurogolf_utils.py"):
        if os.path.exists(cand):
            if importlib.util.find_spec("onnx_tool") is None:  # display-only dep; stub it
                import types
                sys.modules["onnx_tool"] = types.SimpleNamespace(model_profile=lambda *a, **k: None)
            spec = importlib.util.spec_from_file_location("neurogolf_utils", cand)
            UTILS = importlib.util.module_from_spec(spec); spec.loader.exec_module(UTILS)
            break

def score_bytes(name, model_bytes):
    """Official pipeline: sanitize -> ORT(no-opt, profiling) -> verify all examples -> cost."""
    import numpy as np, onnx, onnxruntime
    tn = int(name[4:7])
    task_path = None
    for base in ("/kaggle/input/competitions/neurogolf-2026", "/kaggle/input/neurogolf-2026"):
        p = f"{base}/task{tn:03d}.json"
        if os.path.exists(p): task_path = p; break
    ex = json.load(open(task_path))
    with tempfile.TemporaryDirectory() as td:
        fp = os.path.join(td, name)
        open(fp, "wb").write(model_bytes)
        if os.path.getsize(fp) > UTILS._FILESIZE_LIMIT_IN_BYTES: return None, "filesize"
        s = UTILS.sanitize_model(onnx.load(fp))
        if not s: return None, "sanitize"
        opt = onnxruntime.SessionOptions(); opt.enable_profiling = True
        opt.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_DISABLE_ALL
        opt.profile_file_prefix = os.path.join(td, f"{tn:03d}")
        try: sess = onnxruntime.InferenceSession(s.SerializeToString(), opt)
        except Exception as e: return None, f"load: {e}"
        wrong = 0
        for p_ in ex["train"] + ex["test"] + ex.get("arc-gen", []):
            b = UTILS.convert_to_numpy(p_)
            if not b: continue
            try:
                out = UTILS.run_network(sess, b["input"])
                if not np.array_equal(out, b["output"]): wrong += 1
            except Exception: wrong += 1
        mem, par = UTILS.score_network(s, sess.end_profiling())
        if mem is None or par is None: return None, "unmeasurable"
        if wrong: return None, f"{wrong} examples wrong"
        cost = mem + par
        return cost, f"OK cost={cost} points={max(1.0, 25.0 - math.log(max(1.0, cost))):.3f}"

if UTILS is None:
    print("neurogolf_utils not found in inputs -- overrides will be packaged UNVERIFIED (not recommended)")
accepted = {}
for name, mb in pending.items():
    if UTILS is None:
        accepted[name] = mb; continue
    cost, msg = score_bytes(name, mb)
    print(f"{name}: {msg}")
    if cost is not None:
        accepted[name] = mb
print(f"accepted overrides: {sorted(accepted) if accepted else 'none'}")


In [ ]:
# ============ 4) Package submission.zip (base bundle + accepted overrides) ============
import zipfile
final = dict(MODELS)
final.update(accepted)
with zipfile.ZipFile("/kaggle/working/submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in sorted(final):
        z.writestr(name, final[name])
print(f"submission.zip written: {len(final)} nets"
      + (f" ({len(accepted)} overridden)" if accepted else " (pure 7184.21 base)"))


In [ ]:
# ============ 5) Where to attack: per-task cost table ============
import json, math
COSTS = json.loads('{"1":445,"2":21088,"3":302,"4":5137,"5":6882,"6":153,"7":127,"8":5626,"9":8410,"10":1070,"11":2114,"12":2259,"13":1930,"14":4171,"15":900,"16":10,"17":7801,"18":33865,"19":2798,"20":1526,"21":324,"22":2302,"23":11546,"24":110,"25":11664,"26":200,"27":1310,"28":168,"29":5525,"30":2396,"31":688,"32":910,"33":1639,"34":2365,"35":2261,"36":2177,"37":4746,"38":213,"39":480,"40":374,"41":1789,"42":3110,"43":794,"44":11119,"45":1673,"46":2318,"47":862,"48":1146,"49":386,"50":3911,"51":1803,"52":227,"53":30,"54":26137,"55":10327,"56":34,"57":668,"58":1033,"59":557,"60":1010,"61":1668,"62":2536,"63":2011,"64":13436,"65":641,"66":17607,"67":47,"68":623,"69":3912,"70":3008,"71":2698,"72":421,"73":40,"74":9050,"75":1487,"76":19782,"77":14874,"78":1293,"79":3143,"80":17136,"81":464,"82":190,"83":447,"84":2038,"85":5381,"86":3136,"87":5,"88":1872,"89":7252,"90":3072,"91":3028,"92":6872,"93":3559,"94":2709,"95":345,"96":8887,"97":910,"98":900,"99":1911,"100":543,"101":17578,"102":2527,"103":60,"104":254,"105":2960,"106":626,"107":4078,"108":300,"109":1779,"110":13787,"111":535,"112":1774,"113":30,"114":900,"115":1043,"116":30,"117":4165,"118":35986,"119":1363,"120":910,"121":341,"122":810,"123":1343,"124":1953,"125":3463,"126":605,"127":396,"128":480,"129":80,"130":30,"131":3987,"132":4089,"133":33564,"134":2284,"135":200,"136":1213,"137":3526,"138":13961,"139":904,"140":5,"141":1505,"142":282,"143":838,"144":104,"145":13215,"146":495,"147":532,"148":6007,"149":136,"150":158,"151":900,"152":458,"153":795,"154":2760,"155":142,"156":1744,"157":8329,"158":35327,"159":1568,"160":1334,"161":2570,"162":4068,"163":2345,"164":30,"165":6058,"166":24,"167":148,"168":2239,"169":1670,"170":2545,"171":910,"172":30,"173":17646,"174":8937,"175":1267,"176":240,"177":3621,"178":763,"179":0,"180":210,"181":368,"182":6793,"183":1230,"184":2403,"185":1961,"186":103,"187":34822,"188":1018,"189":913,"190":2527,"191":13330,"192":8621,"193":910,"194":949,"195":921,"196":4538,"197":2602,"198":13566,"199":1837,"200":756,"201":4614,"202":11277,"203":409,"204":11494,"205":12374,"206":4186,"207":236,"208":6083,"209":28440,"210":30,"211":396,"212":2360,"213":1882,"214":767,"215":750,"216":9127,"217":1470,"218":658,"219":17902,"220":900,"221":805,"222":9175,"223":5,"224":2831,"225":1033,"226":1633,"227":100,"228":962,"229":240,"230":900,"231":219,"232":1410,"233":41303,"234":6845,"235":170,"236":237,"237":1901,"238":2495,"239":947,"240":1475,"241":0,"242":432,"243":22650,"244":1179,"245":2743,"246":2753,"247":470,"248":151,"249":287,"250":3925,"251":4920,"252":260,"253":592,"254":1098,"255":22771,"256":2948,"257":409,"258":160,"259":905,"260":2103,"261":200,"262":106,"263":680,"264":5408,"265":4362,"266":350,"267":772,"268":3584,"269":709,"270":3069,"271":1187,"272":402,"273":2160,"274":226,"275":2011,"276":10,"277":3745,"278":6131,"279":5555,"280":6035,"281":2386,"282":446,"283":910,"284":3735,"285":25234,"286":46653,"287":1994,"288":958,"289":742,"290":355,"291":68,"292":434,"293":1286,"294":910,"295":1604,"296":330,"297":1389,"298":135,"299":50,"300":540,"301":1141,"302":1774,"303":1452,"304":1441,"305":636,"306":300,"307":5,"308":1488,"309":10,"310":3287,"311":30,"312":20,"313":188,"314":100,"315":230,"316":426,"317":146,"318":100,"319":21908,"320":770,"321":300,"322":200,"323":2177,"324":8894,"325":2133,"326":30,"327":881,"328":6902,"329":1059,"330":3022,"331":910,"332":563,"333":3227,"334":197,"335":3501,"336":1941,"337":10,"338":16845,"339":140,"340":11039,"341":1429,"342":1585,"343":2037,"344":910,"345":1495,"346":1254,"347":143,"348":2273,"349":21688,"350":9036,"351":1651,"352":460,"353":334,"354":3077,"355":2755,"356":1319,"357":291,"358":4406,"359":4466,"360":340,"361":4838,"362":521,"363":4636,"364":23013,"365":4028,"366":36385,"367":23927,"368":6225,"369":1599,"370":10373,"371":467,"372":710,"373":60,"374":2304,"375":350,"376":186,"377":8505,"378":4489,"379":10794,"380":99,"381":2198,"382":5783,"383":7320,"384":593,"385":30,"386":211,"387":5059,"388":2496,"389":140,"390":3090,"391":159,"392":1951,"393":144,"394":2800,"395":165,"396":11685,"397":2782,"398":5439,"399":91,"400":1866}')
rows = sorted(COSTS.items(), key=lambda kv: -kv[1])
print("30 most expensive tasks (task, cost, points):")
for t, c in rows[:30]:
    print(f"  task{int(t):03d}  cost={c:>6}  points={max(1.0, 25.0 - math.log(max(1, c))):.2f}")
total = sum(max(1.0, 25.0 - math.log(max(1, c))) for c in COSTS.values())
print(f"bundle total (local mirror): {total:.2f}")


## Notes & credits

- Every override you package is verified with the **official** sanitizer/profiler on **all** public examples first — and remember the hidden suite uses fresh seeds of the same [official ARC-GEN generators](https://github.com/google/arc-gen): implement the *rule*, never memorize demos.
- **Bundle lineage (400 nets, total 7186.61):** community shared-archive contributors (thank you — esp. the Consolidated Audit line for 13 server-proven per-task bests) · original agent-crafted circuits (tasks 014/018/021/031/106/108/193/303/330/350) · verified per-task min-merges — all assembled by [**kaggloop**](https://github.com/qurore/kaggloop). Shared per competition rules (OSI-licensed).
- The full agentic pipeline (recon → hypothesis ledger → circuit agents → fresh-seed gates → submission) is open source: **[`github.com/qurore/kaggloop`](https://github.com/qurore/kaggloop)** — ⭐ if you're curious.

**Happy golfing — and if this notebook helped: upvote ⬆**